# SRM Placement Intelligence — Final ML Workflow

This notebook uses the supplied pair-level dataset to analyze skill gaps and evaluate five traditional classifiers for the synthetic suitability label. It does not estimate real hiring outcomes. The target-generation formula is not available in the project files, so feature independence from that formula cannot be established.

The workflow uses a student-grouped holdout and grouped cross-validation, fold-fitted preprocessing, an explicit backend-compatible 20-feature schema, leakage checks, unsupervised analysis, and verified artifact export. Set `SRM_DATA_PATH` or `SRM_SAMPLE_SIZE` before running to override the defaults.

In [ ]:

from google.colab import files
import pandas as pd
import io

uploaded = files.upload()

file_name = next(iter(uploaded))

df = pd.read_csv(io.BytesIO(uploaded[file_name]))

print("Dataset uploaded successfully!")
print("File:", file_name)
print("Dataset shape:", df.shape)

display(df.head())


In [ ]:
from pathlib import Path
import json
import os
import warnings
from collections import Counter

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.base import clone
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    classification_report, confusion_matrix, f1_score, precision_score,
    recall_score, roc_auc_score, silhouette_score,
)
from sklearn.model_selection import (
    GridSearchCV, GroupShuffleSplit, StratifiedGroupKFold, cross_validate,
)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    PLOTTING_AVAILABLE = True
except ImportError:
    PLOTTING_AVAILABLE = False
    print("Plotting libraries unavailable; numerical analysis will continue without charts.")

def show(value):
    try:
        from IPython.display import display as ipython_display
        ipython_display(value)
    except ImportError:
        print(value.to_string(index=False) if hasattr(value, "to_string") else value)

if "__file__" in globals():
    PROJECT_ROOT = Path(__file__).resolve().parents[1]
else:
    cwd = Path.cwd().resolve()
    PROJECT_ROOT = cwd if (cwd / "data").is_dir() else cwd.parent

DATA_PATH = Path("/content/student_job_pairs_rebuilt.csv")
MODEL_DIR = Path(os.environ.get("SRM_MODEL_DIR", PROJECT_ROOT / "model"))
SAMPLE_SIZE = int(os.environ.get("SRM_SAMPLE_SIZE", "20000"))
RANDOM_STATE = 42
TARGET = "suitability_label_synthetic"
NUMERIC_FEATURES = [
    "cgpa", "coding_skill_score", "aptitude_score", "communication_skill_score",
    "logical_reasoning_score", "mock_interview_score", "backlogs", "github_repos",
    "internships_count", "projects_count",
]
CATEGORY_LEVELS = {
    "branch": ["CSE", "ECE", "EEE", "IT"],
    "role_category": ["Business Analyst", "Data Analyst", "Data Engineer", "Data Scientist", "Machine Learning Engineer", "Python Developer"],
    "work_mode": ["Hybrid", "On-site", "Remote"],
}
CATEGORY_BASELINES = {"branch": "CSE", "role_category": "Business Analyst", "work_mode": "Hybrid"}
CATEGORICAL_FEATURES = [
    "branch_ECE", "branch_EEE", "branch_IT",
    "role_category_Data Analyst", "role_category_Data Engineer", "role_category_Data Scientist",
    "role_category_Machine Learning Engineer", "role_category_Python Developer",
    "work_mode_On-site", "work_mode_Remote",
]
FEATURE_NAMES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
EXPECTED_CATEGORY_VALUES = {key: set(values) for key, values in CATEGORY_LEVELS.items()}


## Load and inspect the dataset

A deterministic sample keeps the SVM/KNN and grouped validation practical. Category levels are checked against the full loaded dataset before encoding. Duplicate student-job pairs are counted and removed from the modeling sample.

In [ ]:
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}. Set SRM_DATA_PATH to its location.")
df = pd.read_csv(DATA_PATH)
required_columns = set(NUMERIC_FEATURES + list(CATEGORY_LEVELS) + [TARGET, "student_id", "job_id", "student_skills", "required_skills"])
missing_columns = sorted(required_columns - set(df.columns))
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {missing_columns}")
print("Dataset shape:", df.shape)
print("Target counts (full dataset):")
show(df[TARGET].value_counts(dropna=False).rename_axis("label").reset_index(name="records"))
print("Duplicate student-job pairs:", int(df.duplicated(["student_id", "job_id"]).sum()))

actual_categories = {}
for column, allowed in CATEGORY_LEVELS.items():
    observed = set(df[column].dropna().astype(str).str.strip().unique())
    actual_categories[column] = observed
    if observed != EXPECTED_CATEGORY_VALUES[column]:
        raise ValueError(f"{column} categories do not match the deployment mapping. Observed={sorted(observed)}, expected={sorted(allowed)}")
    if df[column].isna().any():
        raise ValueError(f"Missing categorical values in {column}; baseline imputation must be explicitly reviewed.")
print("Verified full-data category values:", {key: sorted(value) for key, value in actual_categories.items()})

if SAMPLE_SIZE < 1:
    raise ValueError("SRM_SAMPLE_SIZE must be positive.")
df_sampled = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=RANDOM_STATE).reset_index(drop=True)
duplicate_count = int(df_sampled.duplicated(["student_id", "job_id"]).sum())
df_sampled = df_sampled.drop_duplicates(["student_id", "job_id"]).reset_index(drop=True)
df_sampled = df_sampled.dropna(subset=[TARGET, "student_id"]).reset_index(drop=True)
df_sampled[TARGET] = pd.to_numeric(df_sampled[TARGET], errors="raise").astype(int)
if set(df_sampled[TARGET].unique()) != {0, 1}:
    raise ValueError("The sample must contain both target classes encoded as 0 and 1.")
print(f"Sample rows={len(df_sampled):,}; duplicate pairs removed={duplicate_count:,}")
print("Sample missing values:")
show(df_sampled.isna().sum().loc[lambda values: values.gt(0)].rename("missing").reset_index(name="column"))


## Exploratory checks and skill-gap analysis

Skill tokens are stripped, lowercased, and converted to sets. That makes matching case-insensitive and removes duplicates; missing/empty text becomes an empty set. The per-record missing-skill count is required skills minus student skills.

In [ ]:
if PLOTTING_AVAILABLE:
    sns.set_theme(style="whitegrid")
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.countplot(data=df_sampled, x=TARGET, hue=TARGET, legend=False, ax=axes[0])
    axes[0].set_title("Synthetic suitability label distribution")
    sns.histplot(data=df_sampled, x="skill_match_pct", bins=20, ax=axes[1])
    axes[1].set_title("Supplied skill-match percentage")
    plt.tight_layout(); plt.show()
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    for axis, column in zip(axes, CATEGORY_LEVELS):
        sns.countplot(data=df_sampled, y=column, order=CATEGORY_LEVELS[column], ax=axis)
        axis.set_title(column.replace("_", " ").title())
    plt.tight_layout(); plt.show()

def parse_skills(value):
    if not isinstance(value, str):
        return set()
    return {token.strip().casefold() for token in value.split(",") if token.strip()}

df_sampled["parsed_student_skills"] = df_sampled["student_skills"].map(parse_skills)
df_sampled["parsed_required_skills"] = df_sampled["required_skills"].map(parse_skills)
df_sampled["matched_skill_set"] = [a & b for a, b in zip(df_sampled.parsed_student_skills, df_sampled.parsed_required_skills)]
df_sampled["missing_skill_set"] = [required - student for required, student in zip(df_sampled.parsed_required_skills, df_sampled.parsed_student_skills)]
df_sampled["engineered_student_skill_count"] = df_sampled.parsed_student_skills.map(len)
df_sampled["engineered_required_skill_count"] = df_sampled.parsed_required_skills.map(len)
df_sampled["engineered_matched_skill_count"] = df_sampled.matched_skill_set.map(len)
df_sampled["engineered_missing_skill_count"] = df_sampled.missing_skill_set.map(len)
df_sampled["engineered_skill_match_pct"] = np.divide(
    df_sampled.engineered_matched_skill_count * 100.0,
    df_sampled.engineered_required_skill_count,
    out=np.zeros(len(df_sampled), dtype=float),
    where=df_sampled.engineered_required_skill_count.to_numpy() > 0,
)
all_missing_skills = []
for required, student in zip(df_sampled["parsed_required_skills"], df_sampled["parsed_student_skills"]):
    all_missing_skills.extend(required - student)
top_skill_gaps = pd.Series(Counter(all_missing_skills), dtype="int64").sort_values(ascending=False).head(15)
print("Most frequent missing skills in the deterministic sample:")
show(top_skill_gaps.rename_axis("skill").reset_index(name="records"))
if PLOTTING_AVAILABLE and not top_skill_gaps.empty:
    top_skill_gaps.sort_values().plot(kind="barh", figsize=(9, 5), title="Most frequent missing skills")
    plt.xlabel("Student-job records"); plt.tight_layout(); plt.show()


## Explicit 20-feature deployment schema

The ten numerical fields and ten one-hot fields match the Flask contract. Baselines are CSE, Business Analyst, and Hybrid. The encoder is constructed from observed, verified categories and then checked for exact names, order, count, uniqueness, and baseline behavior.

In [ ]:
encoded = pd.DataFrame(index=df_sampled.index)
for column, levels in CATEGORY_LEVELS.items():
    values = df_sampled[column].astype(str).str.strip()
    for category in levels:
        if category == CATEGORY_BASELINES[column]:
            continue
        encoded[f"{column}_{category}"] = values.eq(category).astype("float64")
encoded = encoded[CATEGORICAL_FEATURES]
category_encoding_verified = (
    encoded.columns.tolist() == CATEGORICAL_FEATURES
    and encoded.columns.is_unique
    and all(set(encoded[column].unique()).issubset({0.0, 1.0}) for column in encoded)
)
for field, baseline in CATEGORY_BASELINES.items():
    baseline_rows = df_sampled[field].eq(baseline)
    related = [name for name in CATEGORICAL_FEATURES if name.startswith(field + "_")]
    if not baseline_rows.any() or not encoded.loc[baseline_rows, related].eq(0.0).all().all():
        category_encoding_verified = False
if not category_encoding_verified:
    raise AssertionError("Actual categorical encoding failed schema or baseline checks.")

numeric = df_sampled[NUMERIC_FEATURES].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
X = pd.concat([numeric, encoded], axis=1)
feature_names = X.columns.tolist()
if feature_names != FEATURE_NAMES:
    raise AssertionError(f"Feature order mismatch: {feature_names}")
if len(feature_names) != 20 or len(set(feature_names)) != len(feature_names):
    raise AssertionError("Expected exactly 20 unique deployment features.")
unexpected = set(feature_names) - set(FEATURE_NAMES)
missing_features = set(FEATURE_NAMES) - set(feature_names)
if unexpected or missing_features:
    raise AssertionError(f"Schema mismatch: missing={sorted(missing_features)}, unexpected={sorted(unexpected)}")
y = df_sampled[TARGET].copy()
groups = df_sampled["student_id"].copy()
print("Verified feature names and order:", feature_names)
print("X shape:", X.shape, "; target counts:", y.value_counts().to_dict())


## Target and leakage audit

The project files do not contain the target-generation formula. The target is synthetic, and its independence from predictors cannot be verified. IDs, target-derived fields, eligibility flags, precomputed skill-match outputs, and raw skill text are excluded. Only the documented student profile and job category fields are used. This predicts the supplied synthetic label, not company hiring or actual placement success.

In [ ]:
TARGET_PROXY_TERMS = (
    "suitability_label", "matched_skill", "missing_skill", "skill_match", "match_pct",
    "eligibility", "fresher_friendly", "target", "label", "student_id", "job_id",
)
leakage_candidates = [column for column in X.columns if any(term in column.casefold() for term in TARGET_PROXY_TERMS)]
if leakage_candidates:
    raise AssertionError(f"Potential target proxies entered X: {leakage_candidates}")
for column in ["experience_eligible", "is_fresher_friendly", "skill_match_pct", "matched_skill_count"]:
    if column in df_sampled:
        print(f"Target relationship diagnostic — {column}:")
        show(pd.crosstab(df_sampled[column], y, normalize="index").round(3))
print("Target formula: unavailable in supplied files; feature independence cannot be verified.")
print("Duplicate sampled pairs after removal:", int(df_sampled.duplicated(["student_id", "job_id"]).sum()))


## Student-grouped holdout and cross-validation

`GroupShuffleSplit` keeps all records for a student on one side of the holdout but does not guarantee stratification. We inspect both class distributions and require both classes in each partition. Three-fold `StratifiedGroupKFold` is used on training students only. Each estimator is wrapped with median imputation and scaling so those transformations are fitted inside each CV training fold.

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()
groups_train = groups.iloc[train_idx].copy()
train_students, test_students = set(groups_train), set(groups.iloc[test_idx])
student_overlap = train_students & test_students
if student_overlap:
    raise AssertionError(f"Student leakage across holdout: {len(student_overlap)} students")
if y_train.nunique() != 2 or y_test.nunique() != 2:
    raise ValueError("Both target classes must occur in both partitions; change sample size or split seed.")
print("Training rows/students:", len(X_train), len(train_students))
print("Testing rows/students:", len(X_test), len(test_students))
print("Training classes:", y_train.value_counts().sort_index().to_dict())
print("Testing classes:", y_test.value_counts().sort_index().to_dict())
print("Student overlap:", len(student_overlap))

cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
for fold, (fold_train, fold_valid) in enumerate(cv.split(X_train, y_train, groups=groups_train), start=1):
    shared = set(groups_train.iloc[fold_train]) & set(groups_train.iloc[fold_valid])
    if shared:
        raise AssertionError(f"Student overlap in CV fold {fold}")
    if y_train.iloc[fold_train].nunique() != 2 or y_train.iloc[fold_valid].nunique() != 2:
        raise ValueError(f"Both classes are required in CV fold {fold}")
print("Three-fold stratified grouped CV verified on training groups only.")


## Training, model selection, and final holdout evaluation

All five required algorithms are evaluated consistently. Random Forest tuning uses grouped CV on training data only. The model is selected by mean training-CV suitable-class F1, with precision/recall recorded as context. The holdout is used once for final evaluation and never for tuning or selection.

In [ ]:
def make_pipeline(estimator):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
        ("scaler", StandardScaler()),
        ("model", estimator),
    ])

estimators = {
    "Decision Tree": DecisionTreeClassifier(max_depth=6, class_weight="balanced", random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
    "SVM": SVC(kernel="rbf", class_weight="balanced", random_state=RANDOM_STATE),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Gaussian Naive Bayes": GaussianNB(),
}
scoring = {"f1": "f1", "precision": "precision", "recall": "recall"}
cv_rows = []
tuned_rf_params = None
for name, estimator in estimators.items():
    pipeline = make_pipeline(estimator)
    if name == "Random Forest":
        search = GridSearchCV(
            pipeline,
            param_grid={
                "model__n_estimators": [100, 200],
                "model__max_depth": [None, 12],
                "model__min_samples_leaf": [1, 3],
                "model__class_weight": ["balanced"],
            },
            scoring=scoring, refit="f1", cv=cv, n_jobs=-1, error_score="raise",
        )
        search.fit(X_train, y_train, groups=groups_train)
        best_index = search.best_index_
        params = search.best_params_
        tuned_rf_params = {key.removeprefix("model__"): value for key, value in params.items()}
        row = {
            "Model": name,
            "CV suitable F1": float(search.cv_results_["mean_test_f1"][best_index]),
            "CV suitable precision": float(search.cv_results_["mean_test_precision"][best_index]),
            "CV suitable recall": float(search.cv_results_["mean_test_recall"][best_index]),
            "Best parameters": tuned_rf_params,
        }
    else:
        scores = cross_validate(
            pipeline, X_train, y_train, groups=groups_train, cv=cv,
            scoring=scoring, n_jobs=-1, error_score="raise",
        )
        row = {
            "Model": name,
            "CV suitable F1": float(scores["test_f1"].mean()),
            "CV suitable precision": float(scores["test_precision"].mean()),
            "CV suitable recall": float(scores["test_recall"].mean()),
            "Best parameters": {},
        }
    cv_rows.append(row)
cv_results = pd.DataFrame(cv_rows).sort_values(
    ["CV suitable F1", "CV suitable precision", "CV suitable recall"], ascending=False
).reset_index(drop=True)
best_model_name = str(cv_results.iloc[0]["Model"])
print("Training-only grouped CV results:")
show(cv_results)
print("Selected from training CV:", best_model_name)

# Fit deployment preprocessing once on the complete training partition only.
train_medians = X_train.median().reindex(FEATURE_NAMES)
if train_medians.isna().any():
    raise ValueError("At least one feature has no observed training value.")
X_train_imputed = X_train.fillna(train_medians)
X_test_imputed = X_test.fillna(train_medians)
standard_scaler = StandardScaler().fit(X_train_imputed)
X_train_scaled = standard_scaler.transform(X_train_imputed)
X_test_scaled = standard_scaler.transform(X_test_imputed)
if X_train_scaled.shape[1] != len(FEATURE_NAMES) or not np.isfinite(X_train_scaled).all() or not np.isfinite(X_test_scaled).all():
    raise AssertionError("Training preprocessing produced an invalid feature matrix.")

final_models = {}
test_predictions = {}
test_metrics = []
confusion_matrices = {}
classification_reports = {}
for name, estimator in estimators.items():
    if name == "Random Forest":
        estimator = RandomForestClassifier(**tuned_rf_params, random_state=RANDOM_STATE, n_jobs=-1)
    fitted = clone(estimator).fit(X_train_scaled, y_train)
    predictions = fitted.predict(X_test_scaled)
    final_models[name] = fitted
    test_predictions[name] = predictions
    labels = [0, 1]
    probabilities = None
    if hasattr(fitted, "predict_proba"):
        probabilities = fitted.predict_proba(X_test_scaled)[:, list(fitted.classes_).index(1)]
    elif hasattr(fitted, "decision_function"):
        probabilities = fitted.decision_function(X_test_scaled)
    roc_auc = roc_auc_score(y_test, probabilities) if probabilities is not None else np.nan
    pr_auc = average_precision_score(y_test, probabilities) if probabilities is not None else np.nan
    test_metrics.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, pos_label=1, zero_division=0),
        "Recall": recall_score(y_test, predictions, pos_label=1, zero_division=0),
        "F1 Score": f1_score(y_test, predictions, pos_label=1, zero_division=0),
        "Macro F1": f1_score(y_test, predictions, average="macro", zero_division=0),
        "Weighted F1": f1_score(y_test, predictions, average="weighted", zero_division=0),
        "Balanced Accuracy": balanced_accuracy_score(y_test, predictions),
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc,
    })
    confusion_matrices[name] = confusion_matrix(y_test, predictions, labels=labels)
    classification_reports[name] = classification_report(
        y_test, predictions, labels=labels, target_names=["Unsuitable", "Suitable"],
        zero_division=0, output_dict=True,
    )

evaluation_df = pd.DataFrame(test_metrics)
print("Final held-out test results (synthetic target):")
show(evaluation_df.sort_values("F1 Score", ascending=False).round(4))
for name in final_models:
    print(f"\n{name} classification report:")
    print(classification_report(y_test, test_predictions[name], labels=[0, 1], target_names=["Unsuitable", "Suitable"], zero_division=0))
    print("Confusion matrix (rows=true, columns=predicted):\n", confusion_matrices[name])
if PLOTTING_AVAILABLE:
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    for axis, (_, row) in zip(axes.flat, evaluation_df.iterrows()):
        name = row["Model"]
        sns.heatmap(confusion_matrices[name], annot=True, fmt="d", cmap="Blues",
                    xticklabels=["Unsuitable", "Suitable"], yticklabels=["Unsuitable", "Suitable"], ax=axis)
        axis.set_title(name); axis.set_xlabel("Predicted"); axis.set_ylabel("Actual")
    axes.flat[-1].axis("off")
    plt.tight_layout(); plt.show()
    evaluation_df.set_index("Model")[["Accuracy", "Precision", "Recall", "F1 Score", "Macro F1", "Balanced Accuracy"]].plot(kind="bar", figsize=(13, 6), title="Five-model held-out comparison")
    plt.ylabel("Score"); plt.xticks(rotation=25, ha="right"); plt.tight_layout(); plt.show()


## K-Means and PCA

These exploratory analyses use the training feature matrix only and exclude the suitability label. K-Means candidate selection uses inertia and silhouette score on a fixed training subset; cluster assignments and profiles are descriptive. PCA is fitted on scaled training features and reports explained and cumulative variance.

In [ ]:
# K-Means fit/candidate evaluation is restricted to training data.
silhouette_rows = []
km_sample_size = min(2500, len(X_train_scaled))
km_indices = np.random.default_rng(RANDOM_STATE).choice(len(X_train_scaled), size=km_sample_size, replace=False)
km_validation = X_train_scaled[km_indices]
for k in range(2, 7):
    candidate = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    train_labels = candidate.fit_predict(X_train_scaled)
    sample_labels = train_labels[km_indices]
    silhouette_rows.append({"k": k, "inertia": candidate.inertia_, "silhouette": silhouette_score(km_validation, sample_labels)})
kmeans_diagnostics = pd.DataFrame(silhouette_rows)
selected_k = int(kmeans_diagnostics.sort_values("silhouette", ascending=False).iloc[0]["k"])
kmeans_model = KMeans(n_clusters=selected_k, random_state=RANDOM_STATE, n_init=10).fit(X_train_scaled)
train_clusters = kmeans_model.labels_
cluster_profiles = X_train.assign(cluster=train_clusters).groupby("cluster")[NUMERIC_FEATURES].mean().round(3)
print("K-Means candidate diagnostics:")
show(kmeans_diagnostics)
print("Selected k by training-subset silhouette:", selected_k)
print("Training cluster sizes:", pd.Series(train_clusters).value_counts().sort_index().to_dict())
print("Training cluster profiles (numeric student attributes):")
show(cluster_profiles.reset_index())
if PLOTTING_AVAILABLE:
    kmeans_diagnostics.set_index("k")["inertia"].plot(marker="o", title="K-Means elbow analysis", ylabel="Inertia")
    plt.tight_layout(); plt.show()
    kmeans_diagnostics.set_index("k")["silhouette"].plot(marker="o", title="Training-subset silhouette by k", ylabel="Silhouette")
    plt.tight_layout(); plt.show()

pca = PCA().fit(X_train_scaled)
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)
components_for_80 = int(np.searchsorted(cumulative_variance, 0.80) + 1)
pca_2d = PCA(n_components=2).fit(X_train_scaled)
print("PCA explained variance by component:", np.round(pca.explained_variance_ratio_, 4).tolist())
print("PCA cumulative explained variance:", np.round(cumulative_variance, 4).tolist())
print("Components for at least 80% variance:", components_for_80)
print("Two-component explained variance:", float(pca_2d.explained_variance_ratio_.sum()))
if PLOTTING_AVAILABLE:
    plt.plot(np.arange(1, len(cumulative_variance) + 1), cumulative_variance, marker="o")
    plt.axhline(0.80, color="red", linestyle="--"); plt.xlabel("Number of components")
    plt.ylabel("Cumulative explained variance"); plt.title("PCA cumulative variance (training data)")
    plt.tight_layout(); plt.show()


## Export and reload the Flask-compatible model bundle

All five classifiers are refit on the training partition with the same ordered 20 features. The saved medians and scaler are fitted only on that partition. Each artifact is reloaded and checked for feature count and held-out prediction parity before the configuration is written.

In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
model_files = {
    "Decision Tree": "decision_tree.pkl",
    "Random Forest": "best_random_forest.pkl",
    "SVM": "svm.pkl",
    "KNN": "knn.pkl",
    "Gaussian Naive Bayes": "naive_bayes.pkl",
}
for name, filename in model_files.items():
    model = final_models[name]
    if getattr(model, "n_features_in_", None) != len(FEATURE_NAMES):
        raise AssertionError(f"{name} expects an incorrect feature count.")
    joblib.dump(model, MODEL_DIR / filename)
joblib.dump(train_medians.reindex(FEATURE_NAMES), MODEL_DIR / "train_medians.pkl")
joblib.dump(standard_scaler, MODEL_DIR / "standard_scaler.pkl")

reloaded_models = {name: joblib.load(MODEL_DIR / filename) for name, filename in model_files.items()}
reloaded_medians = joblib.load(MODEL_DIR / "train_medians.pkl")
reloaded_scaler = joblib.load(MODEL_DIR / "standard_scaler.pkl")
if list(reloaded_medians.index) != FEATURE_NAMES:
    raise AssertionError("Reloaded medians have a feature-order mismatch.")
if getattr(reloaded_scaler, "n_features_in_", None) != len(FEATURE_NAMES):
    raise AssertionError("Reloaded scaler has an incorrect feature count.")
reloaded_scaled = reloaded_scaler.transform(X_test.fillna(reloaded_medians))
parity = {}
for name, model in reloaded_models.items():
    if getattr(model, "n_features_in_", None) != len(FEATURE_NAMES):
        raise AssertionError(f"Reloaded {name} feature count mismatch.")
    matches = np.array_equal(final_models[name].predict(X_test_scaled), model.predict(reloaded_scaled))
    parity[name] = bool(matches)
    if not matches:
        raise AssertionError(f"Held-out prediction parity failed for {name}.")
print("Reloaded artifact held-out parity:", parity)

cv_config = cv_results.copy()
cv_config["Best parameters"] = cv_config["Best parameters"].map(lambda value: value if isinstance(value, dict) else {})
config = {
    "feature_names": FEATURE_NAMES,
    "numeric_features": NUMERIC_FEATURES,
    "categorical_features": {
        "branch": {"CSE": None, "ECE": "branch_ECE", "EEE": "branch_EEE", "IT": "branch_IT"},
        "role_category": {"Business Analyst": None, "Data Analyst": "role_category_Data Analyst", "Data Engineer": "role_category_Data Engineer", "Data Scientist": "role_category_Data Scientist", "Machine Learning Engineer": "role_category_Machine Learning Engineer", "Python Developer": "role_category_Python Developer"},
        "work_mode": {"Hybrid": None, "On-site": "work_mode_On-site", "Remote": "work_mode_Remote"},
    },
    "category_encoding_verified": bool(category_encoding_verified),
    "target_column": TARGET,
    "target_limitation": "Synthetic suitability label; generation formula unavailable; not actual placement success.",
    "model_type": best_model_name,
    "selected_model_reason": "Highest mean suitable-class F1 in training-only grouped CV; precision and recall are reported alongside it.",
    "best_random_forest_parameters": tuned_rf_params,
    "preprocessing": "Training-only feature medians followed by StandardScaler; grouped CV pipelines refit preprocessing within each fold.",
    "scikit_learn_version": sklearn.__version__,
    "evaluation_split": "GroupShuffleSplit(test_size=0.20, random_state=42) grouped by student_id; no student overlap; not stratified by splitter.",
    "training_rows": int(len(X_train)), "testing_rows": int(len(X_test)),
    "training_students": int(len(train_students)), "testing_students": int(len(test_students)),
    "training_class_counts": {str(k): int(v) for k, v in y_train.value_counts().sort_index().items()},
    "testing_class_counts": {str(k): int(v) for k, v in y_test.value_counts().sort_index().items()},
    "selected_model": best_model_name,
    "cross_validation": cv_config.to_dict(orient="records"),
    "evaluation_metrics": evaluation_df.astype(object).where(pd.notna(evaluation_df), None).to_dict(orient="records"),
    "confusion_matrices": {name: matrix.tolist() for name, matrix in confusion_matrices.items()},
    "classification_reports": classification_reports,
    "artifact_prediction_parity": parity,
    "kmeans": {"selected_k": selected_k, "diagnostics": kmeans_diagnostics.to_dict(orient="records")},
    "pca": {"explained_variance_ratio": pca.explained_variance_ratio_.tolist(), "components_for_80_percent": components_for_80},
}
config_path = MODEL_DIR / "feature_config.json"
config_path.write_text(json.dumps(config, indent=2, allow_nan=False), encoding="utf-8")
required_artifacts = [*model_files.values(), "train_medians.pkl", "standard_scaler.pkl", "feature_config.json"]
missing_artifacts = [name for name in required_artifacts if not (MODEL_DIR / name).is_file()]
if missing_artifacts:
    raise FileNotFoundError(f"Artifact export incomplete: {missing_artifacts}")
print("Verified artifacts:", [str(MODEL_DIR / name) for name in required_artifacts])
print("Selected model:", best_model_name)
print("Synthetic-target limitation: target-generation formula is unavailable; no actual placement claim is made.")


## Validation report

The execution output above contains the measured split counts, grouped CV scores, held-out metrics, confusion matrices, cluster diagnostics, PCA variance, and artifact parity results. The target is synthetic; the files do not expose its generation formula. Results therefore describe only classification of that supplied label. The final test partition was used only for final evaluation and artifact parity. If execution stops on a missing dependency, category mismatch, class absence, or resource limit, treat the affected status as blocked and do not interpret metrics as complete.

In [ ]:
print("Project root:", PROJECT_ROOT)
print("Model directory:", MODEL_DIR)
print("Model directory exists:", MODEL_DIR.exists())

if MODEL_DIR.exists():
    print("\nFiles inside model directory:")
    for file in MODEL_DIR.iterdir():
        print(file.name)

In [ ]:
import shutil
from pathlib import Path
from google.colab import files

model_dir = Path("/model")
zip_path = "/content/srm_model_artifacts.zip"

shutil.make_archive(
    "/content/srm_model_artifacts",
    "zip",
    root_dir=model_dir
)

files.download(zip_path)